In [9]:
import pandas as pd
import yfinance as yf
import numpy as np
from curl_cffi import requests
import torch
import torch.nn as nn
import torch.nn.functional as F
import CNN_general_stocks
import  joblib

session = requests.Session(impersonate="chrome")
ticker = "NBIS"

data = CNN_general_stocks.get_features_single(ticker, "2y")

next_day_return = data["Next_Day_Return"]
features = data.drop(columns=["Next_Day_Return"])
features_tensor = torch.tensor(features.values, dtype=torch.float32)
features_tensor = features_tensor.to("mps")


Ticker: NBIS


[*********************100%***********************]  1 of 1 completed
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pandas/core/internals/blocks.py:393: RuntimeWarning: divide by zero encountered in log
  result = func(self.values, **kwargs)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pandas/core/internals/blocks.py:393: RuntimeWarning: invalid value encountered in log
  result = func(self.values, **kwargs)


In [10]:
class MLPStockPredictor(nn.Module):
    def __init__(self, input_dim, dropout_rate=0.2):
        super(MLPStockPredictor, self).__init__()
        self.fc1 = nn.Linear(input_dim, 512)
        self.bn1 = nn.BatchNorm1d(512)
        self.dropout1 = nn.Dropout(dropout_rate)
        self.fc2 = nn.Linear(512, 256)
        self.bn2 = nn.BatchNorm1d(256)
        self.dropout2 = nn.Dropout(dropout_rate)
        self.fc3 = nn.Linear(256, 128)
        self.bn3 = nn.BatchNorm1d(128)
        self.dropout3 = nn.Dropout(dropout_rate)
        self.fc4 = nn.Linear(128, 64)
        self.bn4 = nn.BatchNorm1d(64)
        self.dropout4 = nn.Dropout(dropout_rate)
        self.fc5 = nn.Linear(64, 1)

    def forward(self, x):
        x = self.dropout1(F.relu(self.bn1(self.fc1(x))))
        x = self.dropout2(F.relu(self.bn2(self.fc2(x))))
        x = self.dropout3(F.relu(self.bn3(self.fc3(x))))
        x = self.dropout4(F.relu(self.bn4(self.fc4(x))))
        return self.fc5(x).squeeze(-1)
    
model = joblib.load('models/stock_model_1.joblib')
model = model.to('mps')

In [11]:
capital = 100
returns = []

outputs = model(features_tensor)
predictions = outputs.detach().cpu().numpy().flatten()

for i in range(len(data) - 1):
    pred = predictions[i]
    next_ret = next_day_return.iloc[i]

    if(pred <= 3):
        precent_move = 0
    else:
        percent_move = next_ret
    
    returns.append(percent_move)
    capital = capital * (1 + percent_move)



RuntimeError: linear(): input and weight.T shapes cannot be multiplied (53x29 and 30x512)